# CoAtNet — Marrying Convolution and Attention

**Paper:** Z. Dai, H. Liu, Q. V. Le, and M. Tan,
"CoAtNet: Marrying Convolution and Attention for All Data Sizes,"
in *Advances in Neural Information Processing Systems (NeurIPS)*, 2021, vol. 34, pp. 3965–3977.
arXiv: [2106.04803](https://arxiv.org/abs/2106.04803)

> **One-line summary:** A 5-stage staged hybrid network that uses depthwise convolution
> in early stages (local inductive bias) and relative self-attention in later stages
> (global context), outperforming both pure CNNs and ViT on most data regimes.


## Architecture Pipeline (ASCII)

```
Input Image [B, 3, 224, 224]
        │
        ▼
┌───────────────────────────────────────────────────────────────┐
│  S0: Conv Stem  (2 × stride-2 Conv3×3, BN, GELU)             │
│  [B, 64, 56, 56]   pure convolution — builds low-level feats  │
└───────────────────────────────────┬───────────────────────────┘
                                    │ stride-2
┌───────────────────────────────────▼───────────────────────────┐
│  S1: MBConv Blocks  depth=2      [B, 96,  28, 28]             │
│  BN pre-norm → expand 4× → depthwise → SE → project → residual│
└───────────────────────────────────┬───────────────────────────┘
                                    │ stride-2
┌───────────────────────────────────▼───────────────────────────┐
│  S2: MBConv Blocks  depth=3      [B, 192, 14, 14]             │
│  Same as S1, deeper — still conv-dominated features            │
└───────────────────────────────────┬───────────────────────────┘
                     ─── conv→attention transition ───
                                    │ stride-2
┌───────────────────────────────────▼───────────────────────────┐
│  S3: RelAttn Blocks  depth=5     [B, 384,  7,  7]             │
│  Pre-LN → MHSA (+ 2D rel bias) → Pre-LN → FFN                 │
└───────────────────────────────────┬───────────────────────────┘
                                    │ stride-2
┌───────────────────────────────────▼───────────────────────────┐
│  S4: RelAttn Blocks  depth=2     [B, 768,  4,  4]             │
│  Global self-attention with relative position bias             │
└───────────────────────────────────┬───────────────────────────┘
                                    │
                            Global Avg Pool
                                    │
                          pooled_output [B, 768]

─────────────────────────────────────────────────────────────────
 MBConv Block                     RelAttn Block
─────────────────────────────────────────────────────────────────
 x → BN → expand 1×1 → GELU      x → LN → Q K V Linear
       → dw-Conv3×3  → GELU            → Q·K/√d + rel_bias
       → SE (pool + 2FC)               → softmax → · V
       → project 1×1                 → + residual
       → DropPath                  x → LN → FFN (4× expand)
       → + residual                   → + residual
```


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

fig = plt.figure(figsize=(18, 12))

# ── Left panel: 5-stage pipeline ───────────────────────────────────────────
ax1 = fig.add_axes([0.02, 0.05, 0.34, 0.90])
ax1.set_xlim(0, 6)
ax1.set_ylim(0, 13)
ax1.axis('off')
ax1.set_title('CoAtNet-0  Stage Pipeline', fontsize=12, fontweight='bold', pad=8)

stages = [
    ('Input [B, 3, 224, 224]',            12.3, '#e8eaf6', 'lightgrey'),
    ('S0: Conv Stem\n2×stride-2 Conv3×3\n→ [B, 64, 56, 56]',    10.7, '#b3e5fc', '#0288d1'),
    ('S1: MBConv  depth=2\n[B, 96, 28, 28]\n(conv-dominated)',   8.9,  '#c8e6c9', '#2e7d32'),
    ('S2: MBConv  depth=3\n[B, 192, 14, 14]\n(deeper MBConv)',   7.0,  '#dcedc8', '#388e3c'),
    ('S3: RelAttn  depth=5\n[B, 384, 7, 7]\n(attention begins)', 4.9,  '#fff9c4', '#f57f17'),
    ('S4: RelAttn  depth=2\n[B, 768, 4, 4]\n(global context)',   2.9,  '#ffe0b2', '#e65100'),
    ('Global Avg Pool\npooled [B, 768]',  1.2,  '#fce4ec', '#c62828'),
]

for label, y, fc, ec in stages:
    box = FancyBboxPatch((0.3, y - 0.7), 5.4, 1.2,
                         boxstyle='round,pad=0.07', facecolor=fc, edgecolor=ec, linewidth=2.0)
    ax1.add_patch(box)
    ax1.text(3.0, y, label, ha='center', va='center', fontsize=8.5)

for i in range(len(stages) - 1):
    y1 = stages[i][1] - 0.72
    y2 = stages[i+1][1] + 0.72
    ax1.annotate('', xy=(3.0, y2), xytext=(3.0, y1),
                 arrowprops=dict(arrowstyle='->', color='#455a64', lw=1.8))

# Conv↔Attn boundary indicator
ax1.plot([0.1, 5.9], [6.0, 6.0], color='#e53935', lw=1.5, linestyle='--')
ax1.text(3.0, 6.1, 'Conv → Attention transition', ha='center', fontsize=8, color='#e53935')

# ── Middle panel: MBConv block ──────────────────────────────────────────────
ax2 = fig.add_axes([0.37, 0.15, 0.28, 0.75])
ax2.set_xlim(0, 4)
ax2.set_ylim(0, 11)
ax2.axis('off')
ax2.set_title('MBConv Block (S1 / S2)', fontsize=11, fontweight='bold', pad=6)

mbconv = [
    ('Input x [B, C, H, W]', 10.2, '#e8eaf6'),
    ('BN pre-norm', 9.1, '#fff9c4'),
    ('Expand 1×1 Conv (4C)', 7.9, '#b3e5fc'),
    ('GELU', 7.0, '#e8f5e9'),
    ('DepthWise Conv 3×3', 5.9, '#b3e5fc'),
    ('GELU', 5.0, '#e8f5e9'),
    ('SE: pool→FC(C/4)→FC(C)', 4.0, '#ffe0b2'),
    ('Project 1×1 Conv (C)', 3.0, '#b3e5fc'),
    ('DropPath', 2.1, '#fce4ec'),
    ('Residual + x', 1.1, '#dcedc8'),
]
for label, y, color in mbconv:
    box = FancyBboxPatch((0.2, y - 0.42), 3.6, 0.76,
                         boxstyle='round,pad=0.04', facecolor=color, edgecolor='#455a64', linewidth=1.1)
    ax2.add_patch(box)
    ax2.text(2.0, y, label, ha='center', va='center', fontsize=9)

for i in range(len(mbconv) - 1):
    y1 = mbconv[i][1] - 0.44
    y2 = mbconv[i+1][1] + 0.44
    ax2.annotate('', xy=(2.0, y2), xytext=(2.0, y1),
                 arrowprops=dict(arrowstyle='->', color='#455a64', lw=1.3))

# Residual skip arrow
ax2.annotate('', xy=(3.9, 1.1), xytext=(3.9, 10.2),
             arrowprops=dict(arrowstyle='->', color='#1565c0', lw=1.5,
                             connectionstyle='arc3,rad=-0.3'))
ax2.text(3.95, 5.5, 'skip\nconn.', ha='left', fontsize=7.5, color='#1565c0')

# ── Right panel: RelAttn block ─────────────────────────────────────────────
ax3 = fig.add_axes([0.67, 0.15, 0.31, 0.75])
ax3.set_xlim(0, 4)
ax3.set_ylim(0, 11)
ax3.axis('off')
ax3.set_title('RelAttn Block (S3 / S4)', fontsize=11, fontweight='bold', pad=6)

relattn = [
    ('Input x [B, H·W, C]', 10.2, '#e8eaf6'),
    ('LayerNorm (pre-norm)', 9.1,  '#fff9c4'),
    ('Q  K  V  Linear', 7.9, '#bbdefb'),
    ('A = Q·Kᵀ/√d\n+ rel_pos_bias(row, col)', 6.6, '#c5cae9'),
    ('Softmax → · V', 5.4, '#bbdefb'),
    ('Proj Linear', 4.4, '#bbdefb'),
    ('DropPath + residual x', 3.3, '#dcedc8'),
    ('LayerNorm (pre-norm)', 2.3, '#fff9c4'),
    ('FFN: 4× expand → GELU', 1.3, '#ffe0b2'),
    ('DropPath + residual', 0.3, '#dcedc8'),
]
for label, y, color in relattn:
    box = FancyBboxPatch((0.2, y - 0.45), 3.6, 0.80,
                         boxstyle='round,pad=0.04', facecolor=color, edgecolor='#455a64', linewidth=1.1)
    ax3.add_patch(box)
    ax3.text(2.0, y, label, ha='center', va='center', fontsize=8.8)

for i in range(len(relattn) - 1):
    y1 = relattn[i][1] - 0.47
    y2 = relattn[i+1][1] + 0.47
    ax3.annotate('', xy=(2.0, y2), xytext=(2.0, y1),
                 arrowprops=dict(arrowstyle='->', color='#455a64', lw=1.3))

plt.suptitle('CoAtNet — Architecture Diagrams', fontsize=14, fontweight='bold', y=0.995)
plt.savefig('coatnet_architecture.png', dpi=140, bbox_inches='tight')
plt.show()
print('Architecture diagram saved.')


## Key Innovations

### 1. Why Stage Conv → Attention?

The paper analyses the *generalisation* and *capacity* of convolution vs. attention:

- **Convolution** has strong translation equivariance — a great prior for small data.
  Its receptive field grows slowly with depth, which is perfect for low-level edge/texture detection.
- **Self-attention** has global receptive field from layer 1 — powerful but data-hungry
  (needs large datasets to learn position relationships).

CoAtNet places convolution where its inductive bias is *helpful* (S0–S2, small feature maps
with local patterns) and attention where global context is *needed* (S3–S4, small spatial
size → low cost for global attention).

### 2. MBConv Block

Mobile Inverted Bottleneck from MobileNetV2 / EfficientNet, adapted with BN *before* the block:

$$x' = x + \text{Drop}\bigl(\text{Proj}_{1\times1}(\text{SE}(\text{DW}_{3\times3}(\sigma(\text{Expand}_{1\times1}(\text{BN}(x))))))\bigr)$$

- **BN pre-norm** (not post): normalises the input before expansion, which stabilises
  training when the first operation multiplies channels by 4.
- **SE (Squeeze-Excite):** global avg pool → FC(C→C/4) → ReLU → FC(C/4→C) → Sigmoid.
  Learns per-channel importance weights from the global spatial summary.
- **DropPath (Stochastic Depth):** each block is dropped with probability $p_i$
  linearly scheduled from $0$ to $\text{drop\_path\_rate}$:
$$p_i = \frac{i}{\text{total\_blocks}} \cdot \text{drop\_path\_rate}$$

### 3. Relative Multi-Head Self-Attention (RelAttn)

Standard MHSA attention logits:
$$A_{ij} = \frac{\mathbf{q}_i^\top \mathbf{k}_j}{\sqrt{d_k}}$$

CoAtNet adds a learnable 2D relative position bias:
$$A_{ij} = \frac{\mathbf{q}_i^\top \mathbf{k}_j}{\sqrt{d_k}} + B[\Delta r_{ij},\ \Delta c_{ij}]$$

where $B \in \mathbb{R}^{(2H_{\max}-1) \times (2W_{\max}-1)}$ per head, indexed by
$(\Delta r, \Delta c) = (r_i - r_j,\ c_i - c_j)$ clipped to $[-\text{max\_dist}, \text{max\_dist}]$.

### 4. Downsampling Strategy

Between stages: `MaxPool2d(2×2)` halves spatial resolution, then a `1×1 Conv` adjusts channels.
This separates the spatial downsampling (pooling — no learned parameters) from the
channel projection (conv — learned). Simpler and more stable than strided convolutions.

### 5. Model Variants

| Model | Depths [S1,S2,S3,S4] | Dims | Params |
|-------|---------------------|------|--------|
| coatnet_0 | [2,3,5,2] | [96,192,384,768] | ~25 M |
| coatnet_1 | [2,6,14,2] | [96,192,384,768] | ~42 M |
| coatnet_2 | [2,6,14,2] | [128,256,512,1024] | ~75 M |
| coatnet_3 | [2,6,14,2] | [192,384,768,1536] | ~168 M |
| coatnet_7 | [2,6,36,2] | [192,384,768,1536] | ~2.4 B |


## Code Walkthrough

### MBConv (`hale_core.nn.vision.hybrid.coatnet.MBConv`)

```python
class MBConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, expand=4, se_rd=0.25, drop_path=0.0):
        expand_ch = int(in_ch * expand)
        self.conv = nn.Sequential(
            nn.BatchNorm2d(in_ch),          # BN pre-norm
            nn.Conv2d(in_ch, expand_ch, 1), # 1×1 expand
            nn.GELU(),
            nn.Conv2d(expand_ch, expand_ch, 3, stride=stride, padding=1,
                      groups=expand_ch),    # depthwise 3×3
            nn.GELU(),
            SE(expand_ch, rd=int(in_ch * se_rd)),
            nn.Conv2d(expand_ch, out_ch, 1) # project back
        )
        self.use_res = (stride == 1 and in_ch == out_ch)
        self.drop_path = DropPath(drop_path)

    def forward(self, x):
        return x + self.drop_path(self.conv(x)) if self.use_res else self.conv(x)
```

### Relative Position Bias (`CoAtNetRelativeAttention`)

```python
class CoAtNetRelativeAttention(nn.Module):
    def __init__(self, dim, num_heads, max_dist):
        # One bias table per head: [2*max_dist-1, 2*max_dist-1]
        self.rel_bias = nn.Parameter(
            torch.zeros(num_heads, (2*max_dist-1)**2)
        )

    def forward(self, x):          # x: [B, H*W, C]
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        attn = q @ k.transpose(-2, -1) / self.scale

        # Build per-pair (row_diff, col_diff) index
        bias = self.rel_bias[:, self.rel_idx.view(-1)]   # [heads, H*W*H*W]
        bias = bias.view(self.num_heads, H*W, H*W)
        attn = attn + bias
        ...
```

### Backbone Construction

```python
# Build coatnet_0
from hale_core.nn.vision import build_vision_backbone
bb = build_vision_backbone('coatnet_0', img_size=224)

# Stages: [S0_stem, S1_mbconv, S2_mbconv, S3_relattn, S4_relattn]
# feature_maps returns S1..S4 outputs (4 multi-scale maps)
```


In [ ]:
import sys
sys.path.insert(0, '../../src')
import torch
from hale_core.nn.vision import build_vision_backbone

backbone = build_vision_backbone('coatnet_0', img_size=224)
backbone.eval()

print(f'Model: coatnet_0')
print(f'Parameters : {backbone.num_parameters():,}')
print(f'Embed dim  : {backbone.embed_dim}')
print(f'Feature dims: {backbone.feature_dims}')
print()

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = backbone(x)

print('BackboneOutput:')
print(f'  pooled_output : {tuple(out.pooled_output.shape)}')
print(f'  feature_maps  : {[tuple(fm.shape) for fm in out.feature_maps]}')

# Count params per stage
print('\nParameter counts per stage:')
for name, module in [('stem', backbone.stem),
                     ('s1',   backbone.s1),
                     ('s2',   backbone.s2),
                     ('s3',   backbone.s3),
                     ('s4',   backbone.s4)]:
    n = sum(p.numel() for p in module.parameters())
    print(f'  {name:6s}: {n:>10,} params')

# Demonstrate stochastic depth — drop probs increase with depth
print('\nDropPath rates (first 5 blocks):')
cnt = 0
for stage_name in ['s1', 's2', 's3', 's4']:
    stage = getattr(backbone, stage_name)
    for blk in stage.children():
        if hasattr(blk, 'drop_path') and hasattr(blk.drop_path, 'drop_prob'):
            print(f'  block {cnt:02d}: drop_prob={blk.drop_path.drop_prob:.4f}')
            cnt += 1
            if cnt >= 5:
                break
    if cnt >= 5:
        break


## Paper Citation & Related Work

### Primary Reference

> Z. Dai, H. Liu, Q. V. Le, and M. Tan,
> "CoAtNet: Marrying Convolution and Attention for All Data Sizes,"
> in *Advances in Neural Information Processing Systems (NeurIPS)*, 2021, vol. 34, pp. 3965–3977.
> **arXiv:** https://arxiv.org/abs/2106.04803

### Related Work

| Model | Key Idea | Reference |
|-------|----------|----------|
| MobileNetV2 | Inverted residual + depthwise sep. conv | Sandler et al., CVPR 2018 |
| EfficientNet | NAS-optimised MBConv scaling | Tan & Le, ICML 2019 |
| ViT | Pure patch self-attention | Dosovitskiy et al., ICLR 2021 |
| CvT | Conv-Attention hybrid | Wu et al., ICCV 2021 |
| MaxViT | Multi-axis block + grid attn | Tu et al., ECCV 2022 |
| ConvNeXt V2 | Pure conv with global response norm | Woo et al., CVPR 2023 |

### Key Results from Paper

- **CoAtNet-3** achieves **88.52%** ImageNet top-1 without extra data
- **CoAtNet-4** reaches **89.11%** when pretrained on ImageNet-21K
- **CoAtNet-7** achieves **90.88%** with JFT-3B pretraining
- Outperforms ViT-G and EfficientNet-X at equal parameter counts on all dataset sizes
- Key ablation: removing the conv stages (pure attention) drops accuracy by ~1.5% on ImageNet-1K
